# GEMS M7 — Discovery-first research

**Central question.** Can new *geological candidate-generation* methods recover the fault systems that the existing B0 / B1 models miss? Earlier offline work suggests that ~78 % of withheld systems are missed by both models and that the previous candidate generator covered only ~12.5 % of them: candidate **coverage**, not ranking architecture, may be the bottleneck.

**Rules of this notebook**
* No CNN is retrained. B0 / B1 FULL predictions are reloaded read-only (M6 inventory, staged from Drive).
* One protocol for every number: official kernel / α / β, known faults masked **pixel-exactly** (`disc_official`), area budget counted on **eligible** pixels only (template finite, not a known fault, not in the optional ring). The 300 m ring is an *experiment*, not an organizer exclusion.
* Every label-dependent choice for a zone is made on the *other* zones (nested cross-fitting). These zones were inspected in earlier sessions, so they are **not** an untouched test set.
* No number here is a leaderboard score. Oracle numbers read the truth: diagnostic upper bounds only.
* Every section reloads compatible outputs (fingerprints) and writes CSV / Parquet / PNG+SVG / GeoTIFF / GeoPackage / JSON to `MyDrive/GEMS_2/outputs/m7_discovery_first/`.

**Execution path and gates**

| section | what | gate |
|---|---|---|
| 00 | setup, artifact recovery, compatibility | stops on CRS / template mismatch |
| 01 | baseline reconstruction (immutable table) | — |
| 02 | which systems are missed, and why | — |
| 03–04 | multiscale candidates → **coverage oracle** | **GO / NO-GO** (sections 10–12 refuse after NO-GO) |
| 05–07 | DEM pilot: inventory → morphology → incremental value | EXPAND / STOP / INCONCLUSIVE |
| 08–09 | known-fault network, corruption benchmark | learned completer only with headroom |
| 10–12 | candidate features, ranking, rasterisation | needs GO (or `ranking.force`) |
| 13–14 | comparison, decision report | PROMOTE / RETAIN / REJECT / INCONCLUSIVE |
| 15–16 | GeoTIFF export, ≤ 3 leaderboard probes | never uploaded |

Code: `gems_bench/m7/` · configuration: `configs/m7/discovery_first.yaml` · guide: `docs/M7_DISCOVERY_FIRST.md`.

## 00. Project setup and artifact recovery

**Question.** Are all inputs present and mutually compatible (grid, template, folds, splits, features, FULL maps)?

**Method.** Mount Drive, install the repository, open the pipeline context (cached data stages are *validated and staged*, never rebuilt from scratch), load the B0 / B1 FULL OOF + discovery maps through the M6 inventory, check EPSG:32611, grid / template equality, nodata footprints, required features and bands, and list earlier GEMS3 artifacts that can be reused.

**Decision.** Any CRS or template mismatch stops the notebook. A missing optional feature only disables the source that needs it (logged).

### Central configuration
All switches live in the next cell. `M7_CONFIG` overrides `configs/m7/discovery_first.yaml`; unknown keys raise an error naming the closest valid key.

In [ ]:
CONFIG = {                                         # master-pipeline context (data stages are only validated / staged)
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'gems3-discovery-research',               # branch or commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',
    'run_mode': 'PREPARE_DATA',                      # data stages reuse Drive (SKIP_CACHED); M7 never trains CNNs
    'failure_policy': 'interactive',
    'download_missing': False,
    'run_advanced_models': False, 'train_baselines': [], 'run_full_inference': False, 'run_submission': False,
    'feature_stage': 'lazy',
    'use_gpu': True,
    'seed': 2026,
}
DEM_CONFIG = {'enabled': False, 'mode': 'inventory_only', 'download': False}
PRIVATE_REPO = False

M7_CONFIG = {
    'models': ['b0_reference', 'b1_bg_weight'],
    'retrain_existing_cnns': False,                  # True is refused
    'seed': 2026,
    'overwrite_existing': False,                     # True recomputes stages (immutable tables are still kept)
    'sections': {                                    # processing switches (expensive stages off by default)
        's01_baseline': True, 's02_failure_analysis': True, 's03_candidates': True, 's04_oracle': True,
        's05_dem_inventory': True, 's06_dem_morphology': False, 's07_dem_increment': False,
        's08_network_analysis': True, 's09_corruption_benchmark': True,
        's10_candidate_features': True, 's11_candidate_ranking': True, 's12_rasterization': True,
        's13_comparison': True, 's14_decision_report': True,
        's15_submission_export': False, 's16_probe_packages': False,
    },
    'stop_after_oracle_if_inadequate': True,
    'evaluation': {                                  # geographic fold / split selection and budgets
        'folds': 'all', 'splits': 'all', 'budget_domain': 'eligible',
        'area_fracs': [0.0025, 0.005, 0.01, 0.015, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15],
        'ring_px': [0, 1, 2, 3],
    },
    'candidates': {'scales_m': [200, 400, 800, 1600], 'min_persistence': 2, 'min_coherence': 0.35},
    'oracle': {'max_density_km_per_100km2': 80},
    'dem': {'coverage_mode': 'pilot', 'n_pilot_tiles': 8, 'download': False, 'max_gib': 6.0},
    'ranking': {'force': False, 'use_catboost': False, 'run_pu_sensitivity': False},
    'rasterization': {'ring_px': [0, 1, 3]},
    'submission': {'source': 'fold_average', 'area_frac_source': 'geo'},   # 'oof_stitched' without a GPU
    'probes': {'max_probes': 3},
}
SHOW_FIGURES = True

In [ ]:
import base64, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = f"https://github.com/{CONFIG['repo']}.git"

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN') or ''
    hdr = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {hdr}'}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args[:3])} failed:\n{r.stderr.strip()[-1500:]}")
    return r

if not os.path.exists(f'{CODE}/.git'):
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'remote', 'set-url', 'origin', REPO_URL)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
before = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if git('-C', CODE, 'rev-parse', '--verify', '-q', f'refs/remotes/origin/{ref}', check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', ref)
    git('-C', CODE, 'merge', '-q', '--ff-only', f'origin/{ref}')
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{CODE}[colab]'], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
print('code @', HEAD)
if 'gems_bench' in sys.modules and HEAD != before:
    raise RuntimeError('Code changed after gems_bench was imported: Runtime -> Restart session, then run from the top.')

In [ ]:
import json, pathlib
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
from gems_bench.pipeline import Pipeline

pipe = Pipeline.from_notebook(CONFIG, DEM_CONFIG)
pipe.step('bootstrap')
ok, why = pipe.pc.cuda_ready()
print('GPU:', {k: pipe.pc.gpu().get(k) for k in ('cuda', 'name', 'vram_gb')} if ok else f'none ({why}); CPU sections run normally')
# cached data stages: validated on Drive and staged locally (no download, no regeneration of folds / splits)
for s in ('raw_data', 'native_grid', 'spatial_folds', 'discovery_splits', 'feature_store'):
    pipe.step(s)

m7 = pipe.m7(M7_CONFIG)
OUT = m7.ws.root
print('M7 outputs (local):', OUT)
print('M7 outputs (Drive):', m7.cfg['_drive_root'])

def show(names, sub):
    if not SHOW_FIGURES:
        return
    for n in names:
        p = OUT / sub / f'{n}.png'
        display(Image(filename=str(p))) if p.exists() else print('not produced:', sub, n)

def tab(name, sub, n=30):
    t = m7.ws.load_table(name, sub)
    display(t.head(n)) if len(t) else print('empty / not produced:', sub, name)
    return t

In [ ]:
setup = m7.s00_setup()                    # read-only; raises if CRS / template checks fail
print(json.dumps(setup['checks'], indent=1))
inv = m7.g3.m6.inv
display(inv[[c for c in ('model', 'status', 'experiment_id', 'profile', 'n_disc_maps', 'n_checkpoints', 'reasons') if c in inv]])
display(setup['features'])
display(setup['bands'])
print('earlier GEMS3 files available for reuse:', len(setup['legacy']))
display(m7.recover_legacy_full_maps())    # full-region member maps reused only when fingerprints match
display(m7.protocol_keys())

## 01. Reconstruct the strongest existing baseline

**Question.** Under one consistent protocol, how good are B0, B1, their rank fusion, 1-px thinning, and the 300 m known-fault ring?

**Motivation.** Recorded values (geo DTI 0.165 / 0.168 / fusion ≈ 0.178–0.183; pseudo-discovery 0.092 → 0.192 with the ring) came from several protocols. Comparisons are only meaningful on identical units, masks and budget rules.

**Method.** Per-zone rank fusion *w R(B0) + (1 − w) R(B1)*; binary top-*a*, 1-px crest lines, skeleton; ring 0–3 px on `disc_official`; nested selection of (rep, w, frac, ring) on the other zones; `disc_retained_fp` re-scores the same maps. The area fraction counts **eligible** pixels. An area ledger records the positive area after selection, post-operations, ring and scorer mask.

**Hypothesis.** Thin representations beat dilated ones; the ring helps on the proxy, partly because pseudo-discovery under-represents new faults next to known traces.

**Outputs.** `recovered_baselines/baseline_table__<fingerprint>.csv` (**immutable**: never replaced), `baseline_selection.json`, per-zone tables, `recorded_vs_recomputed.csv` (reported, never reconciled), held-out GeoTIFFs, area ledger.

**Decision.** The champion reference for every later comparison is `fusion_thin` (configurable). Ring results are treated as a hypothesis to probe on the leaderboard (probes A / B), not as proof.

In [ ]:
if m7.enabled('s01_baseline'):
    s01 = m7.s01_baseline()
    t = s01['table']
    display(t[[c for c in ('task', 'method', 'dti_crossfit', 'fold_min', 'optimism', 'area_frac_eligible_mean',
                           'dti_zone0', 'dti_zone1', 'dti_zone2') if c in t]])
    display(s01['recorded'])                 # earlier offline numbers vs this protocol (differences are reported)
    tab('area_ledger', 'recovered_baselines', 12)
    show(['baseline_dti', 'baseline_area_ledger'], 'recovered_baselines')

## 02. Baseline geological failure analysis

**Question.** Which faults are invisible to the models, where, and why?

**Method.** Unit = fault *system*. A system is recovered when ≥ 50 % of its scored pixels get TP support ≥ 0.5 from the method's held-out map. Classes: both / B0 only / B1 only / neither. Strata: length, strike sector, linearity / sinuosity, topology (isolated, branched, multi-segment), near-parallel neighbours, distance to known faults, region, magnetic / gravity gradient, slope / relief, cross-modal agreement, DEM coverage. System- and length-weighted recall; prediction density per region; geo (rediscovery) vs disc (discovery) differences. Case studies: nine aligned true-geometry panels per representative system (labels drawn as lines, predictions as filled rasters), with the arrays saved as NPZ.

**Hypothesis.** Missed systems are short, low-contrast, oblique to the dominant strike, or far from known faults.

**Decision criterion.** Strata where *neither* dominates define what the candidate generator (section 03) must reach; they are the targets of the coverage audit.

In [ ]:
if m7.enabled('s02_failure_analysis'):
    s02 = m7.s02_failure_analysis()
    tab('summary', 'failure_analysis/tables')
    st = m7.ws.load_table('recall_by_stratum', 'failure_analysis/tables')
    for s in ('length_bin', 'strike_sector', 'topology', 'dist_known_band', 'magnetic_gradient_tercile'):
        d = st[st.stratum == s]
        if len(d):
            display(d[[c for c in d.columns if c in ('task', 'value', 'n_systems', 'share_both', 'share_b0_only',
                                                    'share_b1_only', 'share_neither') or c.startswith('length_recall_')]])
    tab('prediction_density_by_region', 'failure_analysis/tables', 20)
    prim = m7.cfg['evaluation']['primary_task']
    show([f'overview_classes_{prim}', f'classes_{prim}_length_bin', f'classes_{prim}_strike_sector',
          'length_recall_by_length_bin', 'density_by_region'], 'failure_analysis/figures')
    idx = tab('case_study_index', 'failure_analysis/case_studies', 12)
    for f in idx.figure.head(4) if len(idx) else []:
        show([f[:-4]], 'failure_analysis/case_studies')

## 03. Multiscale geological candidate generator

**Question.** Can label-free structure extraction propose the lines on which missed faults lie?

**Motivation.** Faults are structures, not independent pixels. Potential-field contacts, terrain steps and model crests are *structural* evidence; a fault-specific claim needs agreement between modalities.

**Method.** For each source (magnetic RTP, isostatic gravity, elevation, conductivity, basement depth, magnetic tilt, EDA fused lineaments, B0 / B1 honest maps) and each scale (200–1600 m): edge or Hessian line response → non-maximum suppression → strength + structure-tensor coherence gates → cross-scale persistence → skeleton → endpoint linking with tangent agreement → network labels → junction removal → traced polylines. Fusion removes duplicates (longest first, provenance kept), computes per-modality agreement and flags `multi_evidence` vs `structural_only`. Known-fault tip extensions are added on discovery units. Stable candidate IDs hash source, scale and geometry.

**Leakage control.** No label is read; parameters are fixed in the configuration before section 04 measures coverage, and are not tuned on it.

**Outputs.** `candidate_generation/vectors/*.gpkg`, per-unit NPZ + Parquet tables, inventory, generator log, confidence raster.

In [ ]:
if m7.enabled('s03_candidates'):
    sets = m7.s03_candidates()
    print({k: len(v) for k, v in (sets or {}).items()})
    tab('generator_log', 'candidate_generation/tables', 40)
    inv3 = m7.ws.load_table('candidate_inventory', 'candidate_generation/tables')
    if len(inv3):
        display(inv3.groupby(['task', 'modality', 'evidence_class']).agg(n=('cand_id', 'size'), km=('length_km', 'sum')))
    show(['candidate_inventory_km'], 'candidate_generation/tables')
    if m7.enabled('s02_failure_analysis'):
        m7.s02b_case_studies_with_candidates()   # case-study panels again, now with candidate lines

## 04. Candidate oracle and discovery-coverage audit — **the scientific gate**

**Question.** Does the pool cover substantially more than the ~12.5 % of withheld systems recorded for the earlier generator?

**Oracle definition.** Coverage = candidates rasterised as 1-px lines (value 1) and scored with the official scorer. Recall-density curve: candidates in *label-free* confidence order, cumulated to density caps (km per 100 km²). Oracle DTI: candidates whose mean kernel weight k(d) = max(1 − d / 300 m, 0) to the scored truth is ≥ t (best t per unit, in-sample by construction). The oracle uses labels for diagnosis only and is never deployable.

**Also.** Recall by modality / provenance, by geological stratum, by B0/B1 class (incremental coverage of systems missed by both), and a miss diagnosis naming the 100 m modality that still carries signal on each missed system (or none → DEM / other data).

**Decision (configurable `oracle.go_if`).** GO if pooled system recall at the gate density ≥ 0.30 AND (coverage of missed-by-both ≥ 0.10 OR oracle DTI gain ≥ 0.02 over the best section-01 baseline). On NO-GO, sections 10–12 refuse to run (the next cell prints why and what to widen).

In [ ]:
if m7.enabled('s04_oracle'):
    s04 = m7.s04_oracle()
    sub = 'candidate_generation/oracle'
    gate = tab('gate_decision', sub)
    tab('pooled_curve', sub)
    tab('pooled_oracle', sub)
    tab('incremental', sub)
    show(['recall_density_curves', 'oracle_dti', 'coverage_by_modality', 'coverage_by_b0b1_class', 'coverage_by_stratum'], sub)
    G = m7.gate()
    print('GATE (primary task):', G)
    if G != 'GO':
        print('NO-GO: candidate coverage does not justify a ranker. Missed systems by hint:')
        tab('miss_summary', sub)
        tab('by_stratum', sub, 40)
        print('Widen generation (scales / modalities / gates named above) before sections 10-12; '
              "M7_CONFIG['ranking']['force'] = True overrides (not recommended).")

## 05. DEM inventory, spatial coverage and geological sampling

**Question.** Which 5–10 USGS 1 m DEM tiles form a representative, affordable pilot?

**Method.** Reuses the master pipeline's DEM inventory (`data/raw/dem_1m/inventory/dem_inventory.csv`, else parses the TNM URL JSON). Coverage is never assumed: only listed tiles intersecting the footprint are candidates. Per tile: label density, relief / slope, dominant strike, B0/B1 disagreement, pixels of systems missed by both (section 02). Quotas (random inside each pool, never the best-looking tiles): fault-rich, fault-poor, low-relief basin, mountain front, disagreement, missed-by-both; then stratified random fill. Storage vs free disk is checked; **nothing is downloaded here** (the full collection is ~189 GiB).

**Decision.** Review `dem/pilot/pilot_tiles.csv` and `storage_estimate.json`, then set `M7_CONFIG['dem']['download'] = True` and enable section 06.

In [ ]:
if m7.enabled('s05_dem_inventory'):
    s05 = m7.s05_dem_inventory()
    pil = tab('pilot_tiles', 'dem/pilot')
    print(json.dumps(m7.read_json('dem/pilot/storage_estimate.json'), indent=1, default=str))
    show(['pilot_tiles_map'], 'dem/pilot')

## 06. DEM morphology (tile streaming)

**Method.** Per tile: work-resolution blocks (1 m block-averaged to 2 m) with a halo (boundless reads, nodata kept) → base features (`dem.morphology`: slope, curvatures, scarps, structure tensor) + multidirectional hillshade, TPI, TRI, positive / negative openness, sky-view factor, red-relief differential, multiscale scarp response, directional discontinuity, ridge / valley persistence, edge consistency, local relief → halo cropped → GDAL `sum` / `max` / `q3` aggregation onto the native 100 m window (sub-cell maxima and upper quartiles preserved). Arrays are released after every block. Per-tile GeoTIFFs, band-by-band mosaic, optional 1 m scarp polylines, and a **measured** runtime / storage estimator extrapolated to the inventory.

**Cost.** CPU; minutes per tile. Disabled by default.

In [ ]:
if m7.enabled('s06_dem_morphology'):
    s06 = m7.s06_dem_morphology()
    tab('tile_timing', 'dem/pilot')
    print(json.dumps(m7.read_json('dem/pilot/dem_cost_estimate.json'), indent=1, default=str))

## 07. DEM incremental-value experiment

**Question.** Does 1 m morphology reveal withheld systems that 100 m geophysics does not?

**Method.** A = section-03 geophysical candidates, B = crests of DEM evidence layers, C = A + B fused; all restricted to DEM-covered pixels (a separate protocol, never pooled with full-zone numbers) and capped to the same candidate density. System / length / short-fault recall, low-contrast systems, density of candidates > 1 km from any mapped fault (unlabelled, an upper bound on false candidates), terrain context, Venn decomposition (geophysics only / DEM only / both / neither), systems missed by B0+B1 that DEM recovers.

**Decision.** EXPAND / EXPAND_TARGETED / STOP / INCONCLUSIVE from `dem.go_if` (measured value and projected cost). Surface morphology will not show every hidden fault.

In [ ]:
if m7.enabled('s07_dem_increment'):
    s07 = m7.s07_dem_increment()
    tab('decision', 'dem/candidates')
    tab('pooled', 'dem/candidates')
    v = m7.ws.load_table('venn', 'dem/candidates')
    if len(v):
        display(v.groupby(['task', 'venn']).size().unstack(fill_value=0))
    show(['dem_increment_system_recall', 'dem_venn'], 'dem/candidates')

## 08. Existing fault-network analysis

**Question.** How much new truth is reachable by continuing, bridging or correcting the known map?

**Method.** Skeleton segments, endpoints with outward tangents, branch points, gaps (endpoint pairs pointing at each other), near-parallel pairs (en-echelon / step-overs). Competing hypotheses per endpoint: straight extension, ±15° rotated extensions, evidence-following walk inside a cone, gap bridges; each records how much it depends on prior geometry and its label-free evidence (and DEM alignment when available). On discovery units the known map is the *retained* faults.

**Outputs.** Network tables, full-region hypothesis GeoPackage, per-unit hypotheses, an oracle-style coverage audit per hypothesis family, and the deterministic **map-completion baseline** (hypotheses ranked by evidence, eligible budget, nested frac) saved for section 13.

In [ ]:
if m7.enabled('s08_network_analysis'):
    s08 = m7.s08_network_analysis()
    tab('summary', 'map_completion/candidates')
    a = m7.ws.load_table('completion_coverage_audit', 'map_completion/candidates')
    if len(a):
        display(a.groupby('family')[['tp_w', 'n_truth']].sum().assign(recall_w=lambda d: d.tp_w / d.n_truth.clip(lower=1)))
    tab('completion_baseline_summary', 'map_completion/metrics')

## 09. Synthetic corruption benchmark

**Method.** Systems are assigned to geographic folds **before** corruption; only held-out-fold systems are damaged (endpoint truncation, interior gaps, lateral shift, missing branches, incomplete networks, short faults). The completer sees the damaged map + label-free evidence only. Deterministic completers first; L chosen on the other folds. Metrics: official DTI / recall of the removed geometry, distance to the original trace, directional agreement, added line length > 300 m from any original fault, recall by evidence tercile.

**Decision.** A learned completer (LightGBM on hypotheses, leave-one-fold-out) is trained only if a class clears `map_completion.learned_if`.

In [ ]:
if m7.enabled('s09_corruption_benchmark'):
    s09 = m7.s09_corruption_benchmark()
    tab('corruption_protocol', 'map_completion/corruptions')
    tab('selection', 'map_completion/metrics')
    tab('decision', 'map_completion/metrics')
    tab('learned_completer', 'map_completion/metrics')
    show(['completion_recall'], 'map_completion/metrics')

## 10. Geological candidate feature engineering *(needs GO)*

One row per candidate: geometry (length, strike, curvature, straightness, fragmentation, branching, endpoint connectivity, orientation persistence), geophysical support (|z| evidence, side contrast at 300 / 600 m, gradient coherence, cross-modal agreement, persistence), topographic support (100 m terrain side contrast, slope asymmetry, curvature, relief; 1 m DEM evidence where covered), neural support (B0 / B1 rank mean / max / p90, agreement, disagreement, regional rank), network context (distance to known faults / endpoints, continuation angle, parallel-trace share). Known-fault proximity is a *feature*, never a rule.

Labels are positive-unlabelled: 1 near scored truth, 0 far from it, −1 (dropped, never negative) in between.

In [ ]:
if m7.enabled('s10_candidate_features'):
    s10 = m7.s10_candidate_features()           # returns None (logged) when the oracle gate is closed
    tab('label_balance', 'candidate_ranking/features')
    fg = m7.ws.load_table('feature_groups', 'candidate_ranking/features')
    if len(fg):
        display(fg.groupby('group').size())

## 11. Candidate ranking *(needs GO)*

Controls: **label-free structural ranking** (required) and B0/B1 support. Learned: LightGBM on geometry + geophysics → + topography/DEM → + neural + network → + map-completion candidates in the pool (optional CatBoost, optional PU-weight sensitivity). Leave one zone out; training rows near the held-out zone **and every linked fragment of their networks** are dropped. Scored as 1-px lines at the eligible budget, frac nested. No hyper-parameter search.

**Decision.** A learned ranker is worth keeping only if it beats the label-free control on held-out zones.

In [ ]:
if m7.enabled('s11_candidate_ranking'):
    s11 = m7.s11_candidate_ranking()
    tab('ranking_summary', 'candidate_ranking/models')
    tab('training_audit', 'candidate_ranking/models', 20)
    imp = m7.ws.load_table('feature_importance', 'candidate_ranking/models')
    if len(imp):
        display(imp.groupby('method').head(8))
    show(['ranking_dti'], 'candidate_ranking/models')

## 12. Metric-aware candidate-to-raster conversion *(needs GO)*

Representations: 1-px centerlines, narrow distance-decaying corridors (1–2 px), confidence-dependent width, anti-aliased sub-pixel lines; confidence blends of ranker / label-free / neural ranks; ring 0 / 1 / 3 px (near-known inclusion vs suppression); hybrid = B0/B1 fusion + candidate lines sharing one budget. The 300 m kernel tolerance is not an instruction to draw 300 m-wide faults. Every choice nested; area after masking recorded.

In [ ]:
if m7.enabled('s12_rasterization'):
    s12 = m7.s12_rasterization()
    tab('rasterization_summary', 'candidate_ranking/predictions')
    tab('representation_ablation', 'candidate_ranking/predictions', 40)
    tab('area_after_masking', 'candidate_ranking/predictions', 20)
    show(['representation_ablation'], 'candidate_ranking/predictions')

## 13. Independent evaluation and champion comparison

Every saved held-out map (B0, B1, fusion variants, candidate rankers, representations, hybrid, map-completion baseline) is re-scored on identical units. Each map carries the protocol key it was built under; a mismatch stops the comparison. Reported **separately per protocol and per zone**: geographic DTI per fold, `disc_official` per split, retained-FP alternative, worst zone, weighted short-fault recall, system and length recall, density, regional DTI, sensitivity to known-fault proximity, paired zone deltas and a paired spatial block bootstrap against the champion. DEM-assisted numbers (section 07) stay in their own restricted-protocol table.

With 3 folds / 3 splits these are descriptive paired differences, not significance tests.

In [ ]:
if m7.enabled('s13_comparison'):
    s13 = m7.s13_comparison()
    sub = 'validation/metrics'
    p = tab('method_pooled', sub, 40)
    pz = m7.ws.load_table('paired_zone_deltas', sub)
    if len(pz):
        display(pz.pivot_table(index=['task', 'method'], columns='zone', values='delta_vs_ref'))
    tab('bootstrap', sub, 40)
    tab('known_proximity', sub, 40)
    show(['method_comparison', 'system_recall_comparison', 'paired_zone_deltas', 'known_proximity_recall',
          'regional_dti'], 'validation/figures')

## 14. Automatic decision report

Rule-based verdicts per method against the champion (`decision.promote_if` / `reject_if`): pooled gain, zone win share, worst-zone delta, geographic drop, bootstrap P(Δ > 0), short-fault recall drop — every criterion listed with its value. Then answers to the ten research questions (DEM, coverage, map completion, ranking vs structural score, regional robustness, short faults, which method recovers B0/B1 misses, best recall-density generator, what to submit, next experiment). Report: `reports/M7_DISCOVERY_FIRST_REPORT.md`.

In [ ]:
if m7.enabled('s14_decision_report'):
    s14 = m7.s14_decision_report()
    tab('decision_verdicts', 'reports', 40)
    ans = m7.ws.load_table('research_answers', 'reports')
    for r in ans.itertuples() if len(ans) else []:
        display(Markdown(f'**{r.question}** {r.answer}  \n_{r.evidence}_'))
    rp = OUT / 'reports' / 'M7_DISCOVERY_FIRST_REPORT.md'
    print('report:', rp)

## 15. Competition GeoTIFF export *(off by default)*

Uses the shared, corrected submission code (`gems3.submission`): NaN wherever the official template is NaN (also inside the data footprint), 0 where the template is finite outside the data; the area fraction is counted on eligible full-region pixels (never on known faults, ring or template nodata). Checks: single float32 band, EPSG:32611, template shape / transform / extent, finite values inside the scored footprint, [0, 1], no finite value on template nodata, positive pixels on known faults reported. Each file has a manifest: model id, member checkpoint hashes, input feature manifest, calibration (frac source), generator fingerprint, post-processing, area ledger, final density. Existing files are never overwritten. **Nothing is uploaded.** Fold-average inference needs a GPU unless full-region maps are cached; otherwise set `submission.source = 'oof_stitched'`.

In [ ]:
if m7.enabled('s15_submission_export'):
    s15 = m7.s15_submission_export()
    display(s15)
    for f in sorted((OUT / 'submissions').glob('*.json')):
        j = json.loads(f.read_text())
        if 'area_ledger' in j:
            print(f.name)
            display(pd.DataFrame(j['area_ledger']))

## 16. Leaderboard experiment packages *(off by default)*

At most three probes (3 scored submissions per rolling 7 days):
* **A** champion fusion + thinning, 300 m known-fault ring **on**;
* **B** identical (same representation, weight and number of selected pixels), ring **off** — isolates whether hidden new faults lie next to known traces;
* **C** the best method that passed its section-14 gate, else the conservative fallback.

`submissions/leaderboard_log.csv` is created once (never overwritten). Record each manual upload there. Never invent scores; never tune dozens of variants on public feedback.

In [ ]:
if m7.enabled('s16_probe_packages'):
    s16 = m7.s16_probe_packages()
    display(s16)
    display(pd.read_csv(OUT / 'submissions' / 'leaderboard_log.csv'))

## Run log, outputs and resume

Re-running any cell reuses finished stages (fingerprints in `manifests/units/`). After a disconnect: run the four setup cells, then any section. `M7_CONFIG['overwrite_existing'] = True` recomputes; immutable tables are still kept. Read first: `recovered_baselines/baseline_table__*.csv`, `failure_analysis/tables/summary.csv`, `candidate_generation/oracle/gate_decision.csv`, `reports/M7_DISCOVERY_FIRST_REPORT.md`.

In [ ]:
display(pd.DataFrame(m7.log)) if m7.log else print('no stage ran in this session')
display(m7.summary())
m7.paths()
pipe.push_all()